## EDGE Model Ver. B

### This notebook will contain a variant of the Rocket Richard Baseline Predictor made in rr1.ipynb, that is, this variant will be using General Skater Stats (GSS) + EDGE Stats -- AKA: EDGE Model Ver. B
#### This is done for the sake of model fine tuning and evaluating performance

In [194]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast
from helpersrr import clear_csv, extractPlayerID, placeToStats, fetchSkaterStats, labelWinners, formatEdgeStats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


import importlib
import helpersrr
importlib.reload(helpersrr)

<module 'helpersrr' from '/Users/Joaquin/Documents/GitHub/nhl-trophy-predictor/notebooks/helpersrr.py'>

In [ ]:
#global variables
ranks = False
versionA = False    #version A has no shot details, version B has ALL shot details

client = NHLClient()
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)
third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

../data/formattedwebscraped/rocketrichard.csv


### Milestone 1: Get EDGE Statistics from the API and place them in data/api/EDGEstats

In [196]:
#DO NOT RUN THIS IF COMMENTED -- essentially places relevant EDGE stats files into the data/api/EDGEstats folder
#edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#fetchSkaterStats('20212022',csv=True,edge=True)
#for season in edgeSzns:
#    fetchSkaterStats(season, csv=True, edge=True)
#    print(f"done season {season}")

### Milestone 2: Make the new feature set (GSS + EDGE Stats)

In [197]:
#get the playerIds of the top 3
rr2firstids = placeToStats(rocket_richard[['szn','winner']])
rr2secondids = placeToStats(rocket_richard[['szn','runner_up']])
rr2thirdids = placeToStats(rocket_richard[['szn','finalist']])


In [198]:
#split training sets and testing set
edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
trainingSets = []
testingSet = []

to_drop = ['positionCode','lastName','teamAbbrevs','shGoals','shPoints']    #keep full name instead of last name
for year in edgeSzns:
    if ranks == True:
        #if versionA == True:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=False)
        #else:
        #    modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=False)
    else:
        #if versionA == True:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=False)
        #else:
        #    modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=False)
    
    feature_df = modifiedDf.drop(columns=to_drop)
    feature_df = feature_df.dropna()
    feature_df .loc[feature_df ['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
    feature_df .loc[feature_df ['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
    
    if year == "20252026":  #2025-2026 will be the testing year
        testingSet.append(feature_df)
    else:
        trainingSets.append(feature_df)
print(len(testingSet), len(trainingSets))

masterTraining = pd.DataFrame()
for train in trainingSets:
    masterTraining = pd.concat([masterTraining,train])

masterTesting = pd.DataFrame()
masterTesting = pd.concat([masterTesting,testingSet[0]])

1 4


### Milestone 3: Train model

In [199]:
if ranks == False:
    train_x = masterTraining.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    train_y = masterTraining['rrWinner']
else:
    train_x = masterTesting.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    train_y = masterTesting['rrRank']

model1 = LogisticRegression(max_iter=20000, class_weight="balanced")
model1.fit(train_x,train_y)

#new addition: RandomForest
model2 = RandomForestClassifier(n_estimators=100)
model2.fit(train_x,train_y)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### Milestone 4: Test on EDGE + GSS of 2025-2026

In [200]:
test2025 = masterTesting
if ranks == False:
    test_x = test2025.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    test_y = test2025['rrWinner']
else:
    test_x = test2025.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    test_y = test2025['rrRank']


pred_y1 = model1.predict(test_x)
predictions = pd.Series(pred_y1)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2025.join(predictions)


pred_y2 = model2.predict(test_x)
predictions2 = pd.Series(pred_y2)
predictions2 = predictions2.rename('predictions')
predictions2 = predictions2.to_frame()
results2 = test2025.join(predictions2)


In [201]:
#for model 1

if ranks == False:
    show = results.loc[results['predictions'] == 1]
else:
    show = results.loc[results['predictions'] != 0]
    show = show.dropna()
    hyman = results.loc[results['rrRank'] != 0]

#view influences on the model
feature_names = train_x.columns
coefficients = pd.Series(model1.coef_[0],index=feature_names)
print(coefficients.sort_values())

with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show)  


gamesPlayed                    -0.361129
Outside L Shots                -0.179139
longShots                      -0.167994
skatingSpeed                   -0.124613
assists                        -0.114347
topShotSpeed                   -0.101587
L Point Shots                  -0.096343
L Net Side Shots               -0.082360
R Point Shots                  -0.073175
R Circle Shots                 -0.054829
penaltyMinutes                 -0.047267
ppGoals                        -0.044173
shots                          -0.038729
Low Slot Shots                 -0.037730
R Corner Shots                 -0.035932
Offensive Neutral Zone Shots   -0.031214
totalDistanceSkated            -0.019854
shootsCatches                  -0.014036
longGoals                      -0.012857
L Corner Shots                 -0.007499
timeOnIcePerGame               -0.002509
defensiveZonePctg              -0.002002
faceoffWinPct                  -0.001141
neutralZonePctg                -0.000724
offensiveZonePct

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots,averageTOI,rrWinner,predictions
2,74,42,97,0.51049,7,80,53,1,39,8477492,57,127,1.5875,11,30,20252026,0.15142,1,350,Nathan MacKinnon,1335.7125,146.4342,39.8786,456.0059,7.8713,32,6,149,20,83,19,0.466145,0.18097,0.352885,3,3,16,11,42,77,1,19,11,72,5,38,12,30,0,5,5,22.261875,1.0,1.0


findings/conclusions on the above can be found in commitlog.md

In [202]:
#for model 2 -- uses SHAP to visualize predictions (later)
if ranks == False:
    show = results2.loc[results2['predictions'] == 1]
else:
    show = results2.loc[results2['predictions'] != 0]
    show = show.dropna()
    hyman = results2.loc[results2['rrRank'] != 0]
show

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots,averageTOI,rrWinner,predictions
